# NeuroForge — Quick Ablation (the make-or-break experiment)

A minimal, self-contained notebook that answers the one question that decides the
paper: **does the corrector improve accuracy (field MSE / rho_Cd), not just the
residual — and does the DEQ corrector beat the feed-forward one?**

Install -> download/cache AirfRANS -> train all ablation arms -> print a
mean+/-std table. Set a **GPU** runtime (L4 recommended). Run all, top to bottom.


## 1 - GPU check


In [ ]:
!nvidia-smi -L || echo 'No GPU - set Runtime > Change runtime type > GPU'


## 2 - Install


In [ ]:
import os, sys, subprocess
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[_v] = str(os.cpu_count() or 4)

REPO_URL = 'https://github.com/ali-kin4/neuroforge-cfd.git'  # your repo / fork

def _is_repo(d):
    return bool(d) and os.path.isfile(os.path.join(d, 'pyproject.toml')) \
        and os.path.isdir(os.path.join(d, 'src', 'neuroforge'))

# Portable: works on Colab AND on a local PC. Use the repo we're already inside
# (the PC case — no second clone); otherwise clone it (Colab -> /content, else ./).
_cwd = os.getcwd()
if _is_repo(_cwd):
    REPO_DIR = _cwd
elif _is_repo(os.path.dirname(_cwd)):
    REPO_DIR = os.path.dirname(_cwd)
else:
    REPO_DIR = '/content/neuroforge-cfd' if os.path.isdir('/content') \
        else os.path.abspath('neuroforge-cfd')
    if not _is_repo(REPO_DIR):
        rc = os.system(f'git clone --depth 1 {REPO_URL} "{REPO_DIR}"')
        if rc != 0 or not _is_repo(REPO_DIR):
            raise RuntimeError(f'Clone failed — set REPO_URL/REPO_DIR (tried {REPO_DIR}).')

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data]'], check=True)
for _p in (os.path.join(REPO_DIR, 'src'), REPO_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import neuroforge as nf
from benchmarks.ablation import run_ablation
print('NeuroForge', nf.__version__, 'ready at', REPO_DIR)

## 3 - Storage (repo-local; results are committed)

No Google Drive. The rasterised cache and any checkpoints stay in the
(gitignored) `data/` and `checkpoints/` dirs; the ablation **tables** are written
into the repo's `results/` dir, which **is** tracked — commit and push them back
with `python scripts/push_results.py`.

In [ ]:
# Repo-local storage — no Drive. Results go to results/ so you can push them back.
DATA_ROOT   = os.path.join(REPO_DIR, 'data')            # raw AirfRANS (gitignored, large)
CACHE_DIR   = os.path.join(REPO_DIR, 'data', 'cache')   # rasterised cache (gitignored, large)
CKPT_DIR    = os.path.join(REPO_DIR, 'checkpoints')     # trained .pt (gitignored, large)
RESULTS_DIR = os.path.join(REPO_DIR, 'results')         # tables (COMMITTED -> push back)
for d in (DATA_ROOT, CACHE_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)
print('repo-local storage (results/ is committed; the rest is gitignored):')
print('  CACHE_DIR   =', CACHE_DIR)
print('  RESULTS_DIR =', RESULTS_DIR)

## 4 - Run the ablation

Fast defaults (1 seed, 40 epochs, 150 sims) for a quick directional answer.
For the **full paper run**, set `seeds=(0, 1, 2)`, `epochs=80`, `n_train=400`
(or use `!python benchmarks/ablation.py --source airfrans --task full --seeds 0 1 2 --cache-dir data/cache`).


In [ ]:
TASK = 'full'   # 'full' (800/200) or 'scarce' (smaller, quicker)
try:
    ABL = run_ablation(
        'airfrans', task=TASK, n_train=150, n_val=80, resolution=128,
        seeds=(0,), epochs=40, corrector_epochs=10,
        width=48, modes=20, n_layers=4, batch_size=6,
        root=DATA_ROOT, cache_dir=CACHE_DIR, download=True,
        device='auto', out_dir=RESULTS_DIR, verbose=True,
    )
    print()
    print('Saved table ->', os.path.join(RESULTS_DIR, 'ablation.md'))
    print('Push it back: python scripts/push_results.py')
except Exception:
    import traceback
    err = traceback.format_exc()
    print(err)
    with open(os.path.join(CKPT_DIR, 'last_error.txt'), 'w') as f:
        f.write(err)

## How to read it

- **H1 (corrector helps accuracy):** `backbone + DEQ corrector` should have
  **lower MSE** and **rho_Cd closer to 1** than `backbone`.
- **H2 (residual is a valid trust signal):** `residual_error_spearman` **> 0**.
- **H3 (DEQ >= local):** the DEQ row should match or beat `+ local corrector`.

The full table is in the repo at `results/ablation.md` + `.csv`. Push it back
with `python scripts/push_results.py`, then it's on GitHub to iterate from.